[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/Intelligence-Artificielle-et-Data-Science/blob/main/bloc2_donnees/corrections/seance2_correction.ipynb)

# Séance 2.2 — Agréger, croiser et visualiser

**Correction** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/Intelligence-Artificielle-et-Data-Science/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- répondre à « combien par... ? » avec `groupby`, et calculer plusieurs indicateurs d'un coup
- rassembler plusieurs fichiers avec `merge`, sans perdre ni dupliquer de lignes
- choisir le bon graphique selon la question posée, et le rendre lisible
- repérer ce qu'un graphique cache autant que ce qu'il montre
- conclure une analyse par des recommandations chiffrées

## Correction

Solutions commentées. Comparez avec ce que vous aviez écrit : plusieurs formulations peuvent être correctes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/Intelligence-Artificielle-et-Data-Science/main/bloc2_donnees/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
ventes = pd.read_csv(BASE + "ventes.csv")
clients = pd.read_csv(BASE + "clients.csv")
produits = pd.read_csv(BASE + "produits.csv")

ventes["ca"] = ventes["qte"] * ventes["prix"]     ## le CA de chaque ligne
ventes["date"] = pd.to_datetime(ventes["date"])   ## du texte vers des dates

print(ventes.shape, clients.shape, produits.shape)

---

# Partie 1 — L'échauffement

Le code est déjà écrit : il ne reste que les `____` à remplir. Allez vite, l'essentiel
de la séance est dans la partie 2.

### Exercice 1 — Le meilleur client

> **Votre mission :**
> - Calculer le chiffre d'affaires par client.
> - Mettre l'identifiant du meilleur dans `meilleur_client` et son CA dans `ca_meilleur` (arrondi à 2 décimales).

In [ ]:
ca_client = ventes.groupby("client_id")["ca"].sum()   ## un CA par client

# idxmax() donne l'etiquette du maximum, max() donne sa valeur
meilleur_client = ca_client.idxmax()      ## qui ?
ca_meilleur = round(ca_client.max(), 2)   ## combien ?

print(meilleur_client, ":", ca_meilleur, "euros")

In [ ]:
verifier("1a - meilleur client", meilleur_client == 14911,
         "groupby sur client_id puis sum() sur la colonne ca")
verifier("1b - son chiffre d'affaires", ca_meilleur == 143825.06,
         "idxmax() renvoie l'identifiant, max() renvoie le montant")

### Exercice 2 — Les deux jointures, et le classement des marchés

> **Votre mission :**
> - Joindre `ventes` et `clients` sur `client_id` → `vc`, puis `vc` et `produits` sur `prod_id` → `complet`.
> - **Vérifier après chaque jointure** que le nombre de lignes n'a pas changé.
> - Puis calculer le CA par pays, trié du plus grand au plus petit → `ca_pays`, et mettre celui de la France dans `ca_france` (arrondi à 2 décimales).

In [ ]:
# Le reflexe a ne jamais sauter : un merge peut dupliquer ou faire
# disparaitre des lignes sans rien dire. 45 123 au depart, 45 123 a l'arrivee.
vc = ventes.merge(clients, on="client_id")   ## la cle commune
complet = vc.merge(produits, on="prod_id")   ## deuxieme jointure, autre cle
print(len(ventes), "->", len(vc), "->", len(complet))

# groupby puis tri : le classement des marches, en une ligne
ca_pays = vc.groupby("pays")["ca"].sum().sort_values(ascending=False)

# On accede a une valeur par son etiquette, comme dans un dictionnaire
ca_france = round(ca_pays["France"], 2)

print(ca_pays.head(3).round(2))
print("France :", ca_france)

In [ ]:
verifier("2a - jointure clients sans perte", len(vc) == 45123,
         "un nombre different signale une cle de jointure non unique")
verifier("2b - jointure produits sans perte", len(complet) == 45123,
         "la cle commune entre vc et produits est prod_id")
verifier("2c - CA de la France", ca_france == 133984.8,
         "groupby('pays') puis sum() sur ca, et ca_pays['France']")

### Exercice 3 — Le panier moyen par pays

> **Votre mission :**
> - Pour chaque pays : le CA total (`ca`) et le nombre de **commandes distinctes** (`nb_cmd`).
> - Ajouter une colonne `panier` = CA ÷ nombre de commandes, arrondie à 2 décimales.
> - Mettre le panier moyen irlandais dans `panier_irl`.

In [ ]:
parpays = vc.groupby("pays").agg(
    ca=("ca", "sum"),
    nb_cmd=("cmd_id", "nunique"),   ## des COMMANDES, pas des lignes
)
parpays["panier"] = (parpays["ca"] / parpays["nb_cmd"]).round(2)   ## un ratio

# .loc[ligne, colonne] pour aller chercher une case precise
panier_irl = parpays.loc["Irlande", "panier"]
print(panier_irl)

In [ ]:
verifier("3 - panier moyen irlandais", panier_irl == 1020.33,
         "avec count au lieu de nunique le panier serait ridiculement bas")

### Exercice 4 — Le tableau croisé

> **Votre mission :**
> - Croiser `pays` (en lignes) et `segment` (en colonnes), avec la somme du `ca` → `tableau`.
> - Mettre le CA des clients « premium » français dans `fr_premium` (arrondi à 0 décimale).

In [ ]:
# index = ce qui va en lignes, columns = ce qui va en colonnes
tableau = vc.pivot_table(values="ca", index="pays", columns="segment",
                         aggfunc="sum")   ## somme du ca dans chaque case

fr_premium = round(tableau.loc["France", "premium"], 0)   ## [ligne, colonne]
print(fr_premium)

In [ ]:
verifier("4 - premium francais", fr_premium == 114433.0,
         "index=pays (lignes), columns=segment (colonnes), aggfunc='sum'")

### Exercice 5 — La courbe, avec titre et unité

> **Votre mission :**
> - Tracer le **nombre de commandes distinctes** par mois sous forme de courbe.
> - Titre et unité obligatoires : un graphique sans légende n'est pas un graphique, c'est un dessin.
> - Incliner les étiquettes à 45° et appeler `tight_layout()` : sur un petit écran, sans ça les dates se chevauchent ou sont coupées.
> - Mettre le mois qui compte le plus de commandes dans `mois_cmd`, et le nombre de mois du fichier dans `nb_mois`.

In [ ]:
# nunique() et non count() : une commande de 30 articles reste UNE commande
nb_cmd = complet.groupby(complet["date"].dt.to_period("M"))["cmd_id"].nunique()
nb_cmd.index = nb_cmd.index.astype(str)   ## en texte pour l'affichage

nb_cmd.plot(kind="line", marker="o", figsize=(7, 4))   ## une evolution
plt.title("Nombre de commandes par mois")
plt.ylabel("commandes")     ## la grandeur, pas "valeurs"
plt.xticks(rotation=45)     ## des dates inclinees ne se chevauchent pas
plt.tight_layout()          ## rien ne sera coupe au bord
plt.show()

# Novembre est le mois qui compte le plus de COMMANDES. Le mois le plus
# fort en EUROS est un autre : vous le trouverez a la question 7.
mois_cmd = nb_cmd.idxmax()
nb_mois = len(nb_cmd)   ## 13 : decembre 2010 ET decembre 2011
print(mois_cmd, "|", nb_mois, "mois")

In [ ]:
verifier("5a - mois record en commandes", mois_cmd == "2011-11",
         "nunique() compte les commandes distinctes, count() compterait les lignes")
verifier("5b - nombre de mois", nb_mois == 13,
         "decembre 2010 et decembre 2011 comptent tous les deux")

### Exercice 6 — Les barres, et le bon graphique pour la bonne question

> **Votre mission :**
> - Calculer le CA par **jour de la semaine** dans `ca_jour`, trié du plus petit au plus grand, et le tracer en barres horizontales avec titre et unité. Mettre le jour le plus fort dans `jour_top`.
> - Puis compléter `reponses` avec les quatre types de graphique, **dans l'ordre des questions** :
> - 1. Comment le chiffre d'affaires évolue-t-il dans le temps ? · 2. Quel pays est le plus gros marché ? · 3. Comment les prix sont-ils répartis ? · 4. Les grosses quantités vont-elles avec les prix bas ?

In [ ]:
# .dt.day_name() donne le nom du jour de chaque date
ca_jour = complet.groupby(complet["date"].dt.day_name())["ca"].sum().sort_values()

# barh plutot que bar : les noms de jours tiennent a l'horizontale
ca_jour.plot(kind="barh", figsize=(7, 4))   ## 7 pouces sur 4 : lisible partout
plt.title("Chiffre d'affaires par jour de la semaine")
plt.xlabel("CA (euros)")
plt.tight_layout()
plt.show()

# Six barres seulement : le samedi n'existe pas dans ce fichier.
# Apres un tri croissant, le plus grand est en derniere position.
jour_top = ca_jour.index[-1]

# evolution -> courbe | classement -> barres | repartition -> histogramme
# | relation entre deux grandeurs -> nuage de points
reponses = ["line", "barh", "hist", "scatter"]
print(jour_top, "|", reponses)

In [ ]:
verifier("6a - jour le plus fort", jour_top == "Thursday",
         "sort_values() trie ; apres un tri croissant le plus grand est en position -1")
verifier("6b - le bon graphique pour la bonne question",
         reponses == ["line", "barh", "hist", "scatter"],
         "une evolution, un classement, une repartition, une relation")

---

# Partie 2 — Les questions

Ici, plus de trous : **la cellule sous chaque question est vide**, et c'est à vous
d'écrire le code en entier. C'est exactement ce qu'on vous demandera pour le projet
final, et ce que fait un analyste devant un fichier qu'il découvre.

Certaines questions utilisent une commande que le cours n'a pas montrée. Quand c'est le
cas, l'énoncé vous la donne — savoir se servir d'une commande qu'on vient de lire fait
partie du métier.

> 💡 Pas de vérification automatique dans cette partie. Affichez systématiquement votre
> résultat, et demandez-vous s'il est **plausible** avant de passer à la suite : c'est
> le seul contrôle dont vous disposerez en entreprise.

### Question 7 — La saisonnalité, et le piège de décembre

> **Votre mission :**
> - **Le contexte.** Votre direction prépare le budget de l'an prochain : **où est le chiffre d'affaires, et où sont les risques ?** Les six questions qui suivent répondent à cette question, une pièce à la fois.
> - Calculer le CA par mois dans `ca_mois` (index = le mois en texte, ex. `"2011-10"`), le tracer en **courbe** avec titre et libellé d'axe, et mettre le meilleur mois dans `mois_top`.
> - La courbe montre une chute en décembre. **Avant de conclure**, comptez les jours de décembre présents dans les données → `jours_dec`.
> - Puis répondez : la chute est-elle réelle ? Mettez `True` ou `False` dans `chute_reelle`, et expliquez en commentaire ce que dit vraiment le pic d'octobre.

In [ ]:
ca_mois = complet.groupby(complet["date"].dt.to_period("M"))["ca"].sum()
ca_mois.index = ca_mois.index.astype(str)   ## des etiquettes lisibles

ca_mois.plot(kind="line", marker="o", figsize=(7, 4))
plt.title("Chiffre d'affaires mensuel, decembre 2010 - decembre 2011")
plt.ylabel("CA (euros)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

mois_top = ca_mois.idxmax()   ## 2011-10

dec = complet.query("date >= '2011-12-01'")
jours_dec = dec["date"].dt.day.nunique()   ## des JOURS distincts, pas des lignes

# 8 jours de vente compares a des mois complets : la comparaison n'a
# aucun sens. Il n'y a pas de chute, il y a un mois tronque.
chute_reelle = False

print(mois_top, "|", jours_dec, "jours de decembre |", chute_reelle)

# Le pic d'octobre, lui, est bien reel. Nos clients sont des detaillants :
# ils se reapprovisionnent AVANT Noel, pas pendant.

### Question 8 — Les marchés

> **Votre mission :**
> - CA par pays, top 8, en **barres horizontales** triées, avec titre et unité.
> - Mettre le deuxième marché dans `marche_2`.

In [ ]:
ca_pays8 = complet.groupby("pays")["ca"].sum().nlargest(8)

# barh = barres horizontales : les noms de pays se lisent sans rotation.
# sort_values() croissant car matplotlib dessine de bas en haut.
ca_pays8.sort_values().plot(kind="barh", figsize=(7, 4))
plt.title("Chiffre d'affaires par pays, 2011 (top 8)")
plt.xlabel("CA (euros)")
plt.ylabel("")
plt.tight_layout()
plt.show()

# nlargest est deja trie : position 0 = 1er marche, position 1 = 2e
marche_2 = ca_pays8.index[1]
print(marche_2)

### Question 9 — La concentration client

> **Votre mission :**
> - Calculer le CA par client, puis la part des **10 premiers** dans le CA total → `part_top10` (en %, arrondi à 1 décimale).
> - Compter les clients irlandais → `nb_irl`.
> - Ces deux chiffres ne valent qu'ensemble. Que disent-ils ?

In [ ]:
ca_cli = complet.groupby("client_id")["ca"].sum().sort_values(ascending=False)

part_top10 = round(100 * ca_cli.head(10).sum() / ca_cli.sum(), 1)   ## en %
nb_irl = complet.query("pays == 'Irlande'")["client_id"].nunique()   ## deux !

print(part_top10, "% du CA pour 10 clients |", nb_irl, "clients irlandais")

# 10 clients sur 472 font plus du tiers du chiffre d'affaires, et le
# deuxieme marche du groupe repose sur DEUX comptes. Ce n'est pas un
# marche a developper, c'est une dependance a couvrir.

### Question 10 — Le top produits, et ce qu'il révèle

> **Votre mission :**
> - Afficher les 5 produits qui génèrent le plus de CA → `top_prod`.
> - **Regardez les noms attentivement.** Deux d'entre eux ne sont pas des produits.
> - Mettre leurs deux libellés dans la liste `faux_produits`.

In [ ]:
top_prod = complet.groupby("libelle")["ca"].sum().nlargest(5).round(2)   ## top 5
print(top_prod)

# "Postage" = les frais de port. "Manual" = une saisie manuelle au comptoir
# (c'est le prod_id "M" reperee a la question 8 de la seance 2.1).
# Ce sont des ecritures comptables, pas des articles du catalogue : les
# laisser dans un classement produits fausse toute decision d'assortiment.
faux_produits = ["Postage", "Manual"]

### Question 11 — Le classement corrigé

> **Votre mission :**
> - Refaire le top 5 en excluant `Postage` et `Manual` → `top_reel`.
> - Calculer la part de ces deux lignes dans le CA total → `part_faux` (en %, arrondi à 1 décimale).
> - *Nouveau :* `query("libelle not in @faux_produits")` — le `@` va chercher une variable Python, `not in` inverse l'appartenance.

In [ ]:
# @faux_produits : query() va chercher la variable Python definie plus haut
reels = complet.query("libelle not in @faux_produits")   ## "not in" : l'inverse
top_reel = reels.groupby("libelle")["ca"].sum().nlargest(5).round(2)
print(top_reel)

ca_faux = complet.query("libelle in @faux_produits")["ca"].sum()
part_faux = round(100 * ca_faux / complet["ca"].sum(), 1)
print(part_faux, "% du CA")

# Le vrai meilleur vendeur est le Regency Cakestand, a 23 514 EUR.
# Les frais de port pesaient presque le double.

### Question 12 — Ce que vous en concluez

> **Votre mission :**
> - La cellule ci-dessous rassemble les chiffres des questions 7 à 11. Exécutez-la, puis rédigez votre conclusion dans une **cellule de texte** que vous ajouterez en dessous.
> - **Trois recommandations, chacune appuyée sur un chiffre que vous avez calculé.**
> - Un constat n'est pas une recommandation : « l'Irlande fait 22,7 % du CA » est un constat ; « il faut sécuriser ces deux contrats » est une recommandation.

In [ ]:
print("meilleur mois        :", mois_top)
print("2e marche            :", marche_2, "avec", nb_irl, "clients")
print("part des 10 premiers :", part_top10, "%")
print("faux produits        :", part_faux, "% du CA")

# ---------------------------------------------------------------------
# Reponse attendue (une redaction parmi d'autres) :
#
# 1. RISQUE DE CONCENTRATION — 10 clients sur 472 pesent 37,5 % du chiffre
#    d'affaires, et le 2e marche du groupe (l'Irlande, 22,7 % du CA) repose
#    sur DEUX comptes. Recommandation : securiser ces contrats par des
#    engagements pluriannuels, et ne pas traiter l'Irlande comme un marche
#    a developper mais comme une dependance a couvrir.
#
# 2. SAISONNALITE — le pic est en octobre, pas en decembre : nos clients
#    sont des detaillants qui se reapprovisionnent AVANT Noel.
#    Recommandation : avancer les operations commerciales de six semaines
#    par rapport au calendrier grand public.
#    (Et la "chute" de decembre est un artefact : le fichier s'arrete au 9.)
#
# 3. QUALITE DES DONNEES — 6,2 % du CA est porte par "Postage" et "Manual",
#    qui ne sont pas des produits. Recommandation : les sortir du perimetre
#    avant toute decision d'assortiment, sans quoi les frais de port
#    apparaissent comme notre meilleure vente.
# ---------------------------------------------------------------------